In [1]:
import ast
import gc
import psutil
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

DATA_DIR = Path("data")
MODEL_ID  = "HuggingFaceTB/SmolVLM-500M-Instruct"
LOG_DIR   = Path("logs")

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: NVIDIA GeForce RTX 3090


In [2]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
train_df["choices"] = train_df["choices"].apply(ast.literal_eval)
print(f"Train: {len(train_df):,}")

Train: 3,109


In [3]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

processor.image_processor.do_image_splitting = False
processor.image_processor.size              = {"longest_edge": 512}
processor.image_processor.max_image_size    = {"longest_edge": 512}

yes_token = processor.tokenizer("Yes", add_special_tokens=False)["input_ids"][0]
no_token  = processor.tokenizer("No",  add_special_tokens=False)["input_ids"][0]
print(f"'Yes' token: {yes_token} | 'No' token: {no_token}")

'Yes' token: 10539 | 'No' token: 5230


In [4]:
CKPT = LOG_DIR / "dora_vision_resume_epoch9_20260430_175706"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
base = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
)
base.config.use_cache = False
model = PeftModel.from_pretrained(base, CKPT)
model.eval()
print("Loaded.")

Loading base model...


/home/rohan/pixels/venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
/home/rohan/pixels/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded.


In [5]:
def build_nli_prompt(row, choice_text, max_context_chars=2000):
    text = ""
    if pd.notna(row.get("lecture", None)) and row["lecture"]:
        text += f"Background:\n{row['lecture'][:max_context_chars]}\n\n"
    if pd.notna(row.get("hint", None)) and row["hint"]:
        text += f"Passage:\n{row['hint'][:max_context_chars]}\n\n"
    text += f"Question: {row['question']}\n"
    text += f"Proposed answer: {choice_text}\n"
    text += "Is this the correct answer? Yes or No."
    return f"<|im_start|>User:<image>{text}<end_of_utterance>\nAssistant:"

def score_choices_nli(model, processor, image, row, choices):
    scores = []
    for choice in choices:
        prompt = build_nli_prompt(row, choice)
        inputs = processor(text=[prompt], images=[image],
                          return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
                  for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            logits  = outputs.logits[:, -1, :]
        score = logits[0, yes_token].item() - logits[0, no_token].item()
        scores.append(score)
        del outputs, logits, inputs
        torch.cuda.empty_cache()
    return int(torch.tensor(scores).argmax().item())

In [6]:
correct = 0
errors  = []

for idx in tqdm(range(len(train_df)), desc="Train eval"):
    row   = train_df.iloc[idx]
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
    pred  = score_choices_nli(model, processor, image, row, row["choices"])
    is_correct = pred == int(row["answer"])
    correct   += is_correct
    if not is_correct:
        errors.append({
            "id":      row["id"],
            "pred":    pred,
            "answer":  int(row["answer"]),
            "subject": row["subject"],
            "grade":   row["grade"],
        })

train_acc = correct / len(train_df)
print(f"\nTrain accuracy: {train_acc:.4f} ({correct}/{len(train_df)})")
print(f"Errors: {len(errors)}")

# Breakdown by subject
errors_df = pd.DataFrame(errors)
if len(errors_df) > 0:
    print("\nErrors by subject:")
    print(errors_df["subject"].value_counts())
    print("\nErrors by grade:")
    print(errors_df["grade"].value_counts())

Train eval:   0%|          | 0/3109 [00:00<?, ?it/s]


Train accuracy: 0.9746 (3030/3109)
Errors: 79

Errors by subject:
subject
natural science    77
social science      2
Name: count, dtype: int64

Errors by grade:
grade
grade8    64
grade6    12
grade4     2
grade3     1
Name: count, dtype: int64
